# Dedispersing CHIME/FRB baseband data for FRBayes

This notebook turns one CHIME/FRB beamformed baseband file into the `.npz` file FRBayes reads. It
needs the CHIME/FRB `baseband_analysis` package. The file it writes holds:

| key                     | contents                                                             |
|-------------------------|----------------------------------------------------------------------|
| `power`                 | the dedispersed dynamic spectrum, (n_chan, n_time), good channels only |
| `freq`                  | channel centre frequencies (MHz)                                     |
| `time`                  | sample times (s)                                                     |
| `dm_dedispersion`       | the DM the data were dedispersed with (pc cm⁻³)                      |
| `dedispersion_ref_freq` | the frequency the dedispersion aligned the channels to (MHz)         |
| `dm_dedispersion_err`   | the uncertainty of that DM from DM_phase (pc cm⁻³), if it was found; for reference, e.g. to set the residual DM prior |

`frbayes.data.prepare_burst` reads `dm_dedispersion` and `dedispersion_ref_freq` automatically, and
`frbayes.results.save_run` records them with the fit; FRBayes does not read `dm_dedispersion_err`. The file is written to `examples/data/`, next to the `notebooks/` folder this
notebook is in, where the tutorial reads the example bursts.

## Settings

In [ ]:
import contextlib
import io
import os
import re
import sys
import tempfile
from pathlib import Path

import numpy as np
from baseband_analysis.dev.Morphology_dev import singlebeam_to_input
%matplotlib inline

event_id = "23438043"                     # CHIME/FRB event number
in_file = "/arc/projects/chime_frb/BaseCat2/beamformed_files/FRB20181221A_23438043_beamformed.h5"
out_file = Path("..") / "data" / f"event_{event_id}_BB.npz"   # examples/data, next to examples/notebooks

# The frequency (MHz) the dedispersion aligns the channels to, at which the time axis keeps the
# telescope's clock: singlebeam_to_input calls baseband_analysis.core.dedispersion.coherent_dedisp
# without f_ref, whose default is 400 MHz
DEDISPERSION_REF_FREQ = 400.0

## Load and dedisperse, coherently

On its way through the ionized gas between galaxies, a burst's lower frequencies are delayed more than
its higher ones (dispersion), in proportion to the DM. *Incoherent* dedispersion shifts each channel's
power by the delay at the channel's centre, which leaves the sweep across each channel's own width as
smearing. *Coherent* dedispersion instead works on the recorded voltages, before they are squared into
power: within each channel it undoes the delay at every frequency, by multiplying the voltages' Fourier
transform by the inverse of the gas's effect, so no smearing is left. Only baseband data, which keep
the voltages, allow it.

`singlebeam_to_input` loads the beamformed data, finds a time downsampling at which the burst reaches
`snr_cutoff`, and fits the structure-maximizing DM (`fit_DM=True`). The beamformed files come already
coherently dedispersed at the DM in their `DM_coherent` attribute; without `co_dedisp=True` the
function would only shift the channels incoherently from there to the fitted DM. With
`co_dedisp=True` it dedisperses the voltages coherently again, at the fitted DM, with
`baseband_analysis.core.dedispersion.coherent_dedisp`, which also aligns the channels to 400 MHz, and
forms the power from them. The dedispersion is then entirely coherent, so FRBayes is set up with
`DM_incoherent = 0`; the check after the call confirms it.

With `diagnostic_plots=True`, `singlebeam_to_input` also saves its plots as `.png` and `.pdf` files
in the folder it runs in. It is run in a temporary folder, deleted afterwards, so the plots are shown
here but no files are left behind.

With `co_dedisp=True`, `singlebeam_to_input` returns `DM_err = 0`: it resets the error after
dedispersing again. The DM_phase error survives only in what the function prints ("The structure
maximizing DM is ... +/- ... pc/cc"), so the cell keeps a copy of the printout and reads the error
from it.

In [ ]:
class Tee(io.StringIO):
    # Shows everything printed, as usual, and keeps a copy of it
    def __init__(self, stream):
        super().__init__()
        self.stream = stream
    def write(self, s):
        self.stream.write(s)
        return super().write(s)
    def flush(self):
        self.stream.flush()

print(f"Loading {in_file}")
here = os.getcwd()
printed = Tee(sys.stdout)
# the diagnostic plot files land in a temporary folder and are deleted; the printout is kept
with tempfile.TemporaryDirectory() as scratch, contextlib.redirect_stdout(printed):
    os.chdir(scratch)
    try:
        (data, freq_id, freq, power, valid_channels, DM, DM_err, downsampling_factor, profile, t_res,
         start, full_power) = singlebeam_to_input(
            in_file,
            downsample=4,                 # starting downsampling in time; raised until the burst reaches snr_cutoff
            fit_DM=True,                  # fit the structure-maximizing DM
            co_dedisp=True,               # then dedisperse the voltages coherently at that DM
            diagnostic_plots=True,
            snr_cutoff=12,
        )
    finally:
        os.chdir(here)

# The DM the power was coherently dedispersed at should now be the fitted DM; any difference was
# removed only incoherently, and is the DM_incoherent FRBayes would need
DM_coherent = float(data["tiedbeam_power"].attrs["DM_coherent"])
print(f"fitted DM {float(DM):.4f}, coherently dedispersed at {DM_coherent:.4f} pc cm^-3: "
      f"DM_incoherent = {float(DM) - DM_coherent:+.4f}")

# The DM_phase result, from the printout (the returned DM_err is reset to 0); the last one printed
def last_number(pattern, group=1):
    found = re.findall(pattern, printed.getvalue())
    if not found:
        return None
    text = found[-1] if isinstance(found[-1], str) else found[-1][group - 1]   # one group: findall gives strings
    try:
        return float(text.strip("[](),"))
    except ValueError:
        return None

DM_snr = last_number(r"S/N maximizing DM is\s*(\S+)\s*pc/cc")
DM_struct = last_number(r"structure maximizing DM is\s*(\S+)\s*pc/cc\s*\+/-\s*(\S+)\s*pc/cc", 1)
DM_phase_err = last_number(r"structure maximizing DM is\s*(\S+)\s*pc/cc\s*\+/-\s*(\S+)\s*pc/cc", 2)
# An error of exactly 0 (or not finite) is not an uncertainty: DM_phase did not produce one
if DM_phase_err is not None and not (np.isfinite(DM_phase_err) and DM_phase_err > 0):
    DM_phase_err = None
print(f"DM_phase error: {DM_phase_err:.3e} pc cm^-3" if DM_phase_err is not None
      else "DM_phase gave no usable error (missing, 0 or not finite); none will be saved")
if DM_snr is not None and DM_struct is not None and DM_struct == DM_snr:
    print("warning: the structure-maximizing DM equals the S/N-maximizing DM exactly, so DM_phase "
          "probably failed and the DM used is the one that maximizes S/N, not structure")

## Build the arrays and check them

FRBayes needs the bad channels removed, not zeroed or masked, and times in seconds. The checks below
drop any channel that is entirely zero or holds a NaN, and print the time resolution and segment
length so the units can be checked by eye.

In [ ]:
power = np.asarray(power, dtype=float)
freq = np.asarray(freq, dtype=float)
n_time = power.shape[1]
time = start + t_res * np.arange(n_time)                   # seconds

bad = ~np.isfinite(power).all(axis=1) | np.all(power == 0, axis=1)
if bad.any():
    print(f"dropping {bad.sum()} channels that are all zero or hold a NaN")
    power, freq = power[~bad], freq[~bad]

df = np.diff(np.sort(freq))
print(f"{power.shape[0]} channels, {freq.min():.1f}-{freq.max():.1f} MHz, "
      f"{np.sum(df > 1.5 * np.median(df))} gaps where bad channels were removed")
print(f"{n_time} samples of {t_res * 1e6:.2f} us (downsampling {downsampling_factor}), "
      f"segment {n_time * t_res * 1e3:.2f} ms")
print(f"dedispersed with DM {float(DM):.6f} pc cm^-3"
      + (f" +/- {DM_phase_err:.2e} (DM_phase)" if DM_phase_err is not None else ""))

## Save

In [ ]:
out_file.parent.mkdir(parents=True, exist_ok=True)
arrays = dict(power=power, freq=freq, time=time,
              dm_dedispersion=float(DM), dedispersion_ref_freq=float(DEDISPERSION_REF_FREQ))
if DM_phase_err is not None:
    arrays["dm_dedispersion_err"] = DM_phase_err
np.savez_compressed(out_file, **arrays)
print(f"saved {out_file}: {', '.join(arrays)}")